In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Copy the dataset from your Drive to Colab's local storage
!cp -r '/content/drive/MyDrive/Solar Monitoring/Dataset/Soiling_DS_v2_14' '/content/'

# 3. Clone the baseline YOLOv5 repository and install dependencies
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
!pip install -qr requirements.txt

import torch
print(f"Setup complete. Using torch {torch.__version__} ({torch.cuda.get_device_properties(0).name if torch.cuda.is_available() else 'CPU'})")

Mounted at /content/drive
cp: cannot stat '/content/drive/MyDrive/Solar Monitoring/Dataset/Soiling_DS_v2_14': No such file or directory
Cloning into 'yolov5'...
remote: Enumerating objects: 17968, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 17968 (delta 72), reused 25 (delta 22), pack-reused 17870 (from 3)
Receiving objects: 100% (17968/17968), 17.13 MiB | 23.23 MiB/s, done.
Resolving deltas: 100% (12229/12229), done.
/content/yolov5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 9.7 MB/s eta 0:00:00
Setup complete. Using torch 2.10.0+cpu (CPU)


In [ ]:
import yaml

# Define the new Linux paths for Colab
yaml_data = {
    'train': '/content/Soiling_DS_v2_14/train/images',
    'val': '/content/Soiling_DS_v2_14/valid/images',
    'test': '/content/Soiling_DS_v2_14/test/images',
    'nc': 2,
    'names': ['Dust', 'bird-droppings']
}

# The path to your data.yaml file inside Colab
yaml_file_path = '/content/Soiling_DS_v2_14/data.yaml'

# Overwrite the file with the new data
with open(yaml_file_path, 'w') as f:
    yaml.dump(yaml_data, f, default_flow_style=False)

print("data.yaml successfully updated for Colab!")

FileNotFoundError: [Errno 2] No such file or directory: '/content/Soiling_DS_v2_14/data.yaml'

In [ ]:
# Read and print the updated data.yaml file
with open('/content/Soiling_DS_v2_14/data.yaml', 'r') as f:
    print(f.read())

In [ ]:
%%writefile -a /content/yolov5/models/common.py

# --- Custom CBAM Module for SDS-YOLO ---
import torch

class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc1   = nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False)
        self.relu1 = nn.ReLU()
        self.fc2   = nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc2(self.relu1(self.fc1(self.avg_pool(x))))
        max_out = self.fc2(self.relu1(self.fc1(self.max_pool(x))))
        out = avg_out + max_out
        return self.sigmoid(out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        assert kernel_size in (3, 7), 'kernel size must be 3 or 7'
        padding = 3 if kernel_size == 7 else 1
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg_out, max_out], dim=1)
        x = self.conv1(x)
        return self.sigmoid(x)

class CBAM(nn.Module):
    def __init__(self, c1, c2, ratio=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.ca = ChannelAttention(c1, ratio)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        out = x * self.ca(x)
        result = out * self.sa(out)
        return result
# ---------------------------------------

In [ ]:
import re

file_path = '/content/yolov5/models/yolo.py'
with open(file_path, 'r') as f:
    content = f.read()

# Add CBAM to the import statement
content = content.replace('from models.common import ', 'from models.common import CBAM, ')

# Add CBAM to the list of modules the parser builds
content = re.sub(r'(C3x, RepC3, C2f, C2fPSA)', r'\1, CBAM', content)

with open(file_path, 'w') as f:
    f.write(content)

print("yolo.py patched successfully to recognize CBAM!")

In [ ]:
%%writefile /content/yolov5/models/sds-yolo.yaml

# parameters
nc: 2  # number of classes
depth_multiple: 0.33  # model depth multiple
width_multiple: 0.50  # layer channel multiple

# Custom anchors from the research paper
anchors:
  - [15, 20, 42, 34, 119, 127]     # P3: Bird droppings
  - [238, 179, 89, 591, 295, 377]  # P4: Dust

# YOLOv5 backbone
backbone:
  [[-1, 1, Conv, [64, 6, 2, 2]],  # 0-P1/2
   [-1, 1, Conv, [128, 3, 2]],    # 1-P2/4
   [-1, 3, C3, [128]],
   [-1, 1, Conv, [256, 3, 2]],    # 3-P3/8
   [-1, 6, C3, [256]],
   [-1, 1, Conv, [512, 3, 2]],    # 5-P4/16
   [-1, 9, C3, [512]],
   [-1, 1, Conv, [1024, 3, 2]],   # 7-P5/32
   [-1, 3, C3, [1024]],
   [-1, 1, SPPF, [1024, 5]],      # 9
  ]

# SDS-YOLO head (Reduced to 2 detection heads)
head:
  [[-1, 1, Conv, [512, 1, 1]],
   [-1, 1, nn.Upsample, [None, 2, 'nearest']],
   [[-1, 6], 1, Concat, [1]],  # cat backbone P4
   [-1, 3, C3, [512, False]],  # 13

   [-1, 1, Conv, [256, 1, 1]],
   [-1, 1, nn.Upsample, [None, 2, 'nearest']],
   [[-1, 4], 1, Concat, [1]],  # cat backbone P3
   [-1, 3, C3, [256, False]],  # 17 (P3/8-small)

   # Attention Module 1
   [-1, 1, CBAM, [256, 256]],  # 18

   [-1, 1, Conv, [256, 3, 2]],
   [[-1, 14], 1, Concat, [1]], # cat head P4
   [-1, 3, C3, [512, False]],  # 21 (P4/16-medium)

   # Attention Module 2
   [-1, 1, CBAM, [512, 512]],  # 22

   [[18, 22], 1, Detect, [nc, anchors]],  # Detect(P3, P4)
  ]

In [ ]:
file_path = '/content/yolov5/models/yolo.py'
with open(file_path, 'r') as f:
    content = f.read()

# Fix the broken import syntax
content = content.replace('from models.common import CBAM, (', 'from models.common import (\n    CBAM,')

with open(file_path, 'w') as f:
    f.write(content)

print("Syntax error fixed!")

In [ ]:
%%writefile /content/yolov5/models/sds-yolo.yaml

# parameters
nc: 2  # number of classes
depth_multiple: 0.33  # model depth multiple
width_multiple: 0.50  # layer channel multiple

# Custom anchors from the research paper
anchors:
  - [15, 20, 42, 34, 119, 127]     # P3: Bird droppings
  - [238, 179, 89, 591, 295, 377]  # P4: Dust

# YOLOv5 backbone
backbone:
  [[-1, 1, Conv, [64, 6, 2, 2]],  # 0-P1/2
   [-1, 1, Conv, [128, 3, 2]],    # 1-P2/4
   [-1, 3, C3, [128]],
   [-1, 1, Conv, [256, 3, 2]],    # 3-P3/8
   [-1, 6, C3, [256]],
   [-1, 1, Conv, [512, 3, 2]],    # 5-P4/16
   [-1, 9, C3, [512]],
   [-1, 1, Conv, [1024, 3, 2]],   # 7-P5/32
   [-1, 3, C3, [1024]],
   [-1, 1, SPPF, [1024, 5]],      # 9
  ]

# SDS-YOLO head (Reduced to 2 detection heads)
head:
  [[-1, 1, Conv, [512, 1, 1]],
   [-1, 1, nn.Upsample, [None, 2, 'nearest']],
   [[-1, 6], 1, Concat, [1]],  # cat backbone P4
   [-1, 3, C3, [512, False]],  # 13

   [-1, 1, Conv, [256, 1, 1]],
   [-1, 1, nn.Upsample, [None, 2, 'nearest']],
   [[-1, 4], 1, Concat, [1]],  # cat backbone P3
   [-1, 3, C3, [256, False]],  # 17 (P3/8-small)

   # Attention Module 1 (Manually scaled down: 256 * 0.5 = 128)
   [-1, 1, CBAM, [128, 128]],  # 18

   [-1, 1, Conv, [256, 3, 2]],
   [[-1, 14], 1, Concat, [1]], # cat head P4
   [-1, 3, C3, [512, False]],  # 21 (P4/16-medium)

   # Attention Module 2 (Manually scaled down: 512 * 0.5 = 256)
   [-1, 1, CBAM, [256, 256]],  # 22

   [[18, 22], 1, Detect, [nc, anchors]],  # Detect(P3, P4)
  ]

In [ ]:
!sed -i 's/deterministic=True/deterministic=False/g' /content/yolov5/train.py

In [ ]:
# Start training the custom SDS-YOLO model
!python train.py \
  --img 640 \
  --batch 16 \
  --epochs 200 \
  --data /content/Soiling_DS_v2_14/data.yaml \
  --cfg models/sds-yolo.yaml \
  --weights yolov5s.pt \
  --patience 15 \
  --name sds_yolo_soiling

In [ ]:
# Create a folder in your Drive to store the model
!mkdir -p '/content/drive/MyDrive/Dataset/Model_Weights'

# Copy the best weights to your Drive (Note: using soiling4 based on your screenshot)
!cp /content/yolov5/runs/train/sds_yolo_soiling4/weights/best.pt /content/drive/MyDrive/Dataset/Model_Weights/sds_yolo_best.pt

print("Weights safely backed up to Google Drive!")

In [ ]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt

def extract_panels(image_path, output_dir="extracted_panels"):
    os.makedirs(output_dir, exist_ok=True)

    # Read image
    img = cv2.imread(image_path)
    if img is None:
        print("Image not found!")
        return

    # 1. Smoothing: Gaussian filter to reduce noise [cite: 268]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    # 2 & 3 & 4. Gradient, Non-Max Suppression, and Hysteresis Thresholding (Canny does all this) [cite: 270, 272, 273]
    edges = cv2.Canny(blurred, 50, 150)

    # Find contours (the borders of the panels)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    saved_count = 0
    for i, contour in enumerate(contours):
        # Filter out small noise - only keep large rectangular contours
        area = cv2.contourArea(contour)
        if area > 5000:  # Adjust this threshold based on your drone image resolution
            x, y, w, h = cv2.boundingRect(contour)

            # Crop the panel (Region of Interest) [cite: 275]
            roi = img[y:y+h, x:x+w]

            # Save the isolated panel
            save_path = f"{output_dir}/panel_{saved_count}.jpg"
            cv2.imwrite(save_path, roi)
            saved_count += 1

    print(f"Successfully extracted {saved_count} panels to '{output_dir}/'")

# Try it out! Upload a full drone image to Colab and replace 'your_test_image.jpg'
# extract_panels('your_test_image.jpg')

In [ ]:
!python detect.py \
  --weights /content/drive/MyDrive/Dataset/Model_Weights/sds_yolo_best.pt \
  --img 640 \
  --conf 0.25 \
  --source /content/Soiling_DS_v2_14/test/images/ \
  --save-txt \
  --save-crop

In [ ]:
import os
from google.colab import files
import matplotlib.pyplot as plt
import cv2

# 1. Upload the unseen image
print("Please upload an image for testing:")
uploaded = files.upload()
img_path = list(uploaded.keys())[0]

# 2. Run inference using the saved SDS-YOLO weights
!python detect.py \
  --weights /content/drive/MyDrive/Dataset/Model_Weights/sds_yolo_best.pt \
  --img 640 \
  --conf 0.25 \
  --source {img_path} \
  --project runs/detect \
  --name unseen_testing \
  --exist-ok \
  --save-txt

# 3. Find the result directory
res_dir = '/content/yolov5/runs/detect/unseen_testing'
img_name = os.path.basename(img_path)
result_img_path = os.path.join(res_dir, img_name)
label_path = os.path.join(res_dir, 'labels', img_name.replace('.jpg', '.txt').replace('.png', '.txt').replace('.jpeg', '.txt'))

# 4. Display Summary
print('\n' + '='*30)
print("SUMMARY RESULTS")
print('='*30)
if os.path.exists(label_path):
    with open(label_path, 'r') as f:
        lines = f.readlines()
        classes = {'0': 'Dust', '1': 'Bird-Droppings'}
        counts = {}
        for line in lines:
            cls_id = line.split()[0]
            label = classes.get(cls_id, 'Unknown')
            counts[label] = counts.get(label, 0) + 1

        for label, count in counts.items():
            print(f"{label}: {count} instance(s) detected.")
else:
    print("No soiling detected in this image.")

# 5. Show result image
if os.path.exists(result_img_path):
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(cv2.imread(result_img_path), cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title(f"SDS-YOLO Detection: {img_name}")
    plt.show()
else:
    print("Error: Could not find output image.")